In [10]:
from langchain_ollama import OllamaEmbeddings

# 创建向量模型,我们今天使用ollama
ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)

# 初始化向量数据库客户端对象
from pymilvus import MilvusClient
from app.core.config import settings

# 初始化向量数据库
milvus_client = MilvusClient(uri=settings.rag.milvus_url)

# Collection名称:集合,指的就是表名字
collection_name = "my_collection_1"

In [11]:
import json
# 从向量数据库中导入向量匹配类
from pymilvus import AnnSearchRequest
# 定义混合检索函数
def hybrid_search(user_question,ranker):
    """
    创建两个查询对象,将结果混合在一起重新进行权重计算,再排序
    :param user_question: 用户提示词
    :param ranker: 每种检索的排序权重
    :return: 返回新的权重计算后的匹配结果
    """
    # 用户提示词转稠密向量
    user_vector = ollama_embeddings.embed_query(user_question)

    # 创建稠密匹配对象
    dense_query = AnnSearchRequest(
        # 不需要输入集合名,设置匹配参数
        data=[user_vector],
        # 匹配稠密向量字段
        anns_field='dense',
        # 查询使用余弦相似度算法
        param={"metric_type": "COSINE"},
        # 最大匹配三条
        limit=3
    )

    # 创建稀疏向量匹配对象
    sparse_query = AnnSearchRequest(
        data=[user_question],
        # 匹配稠密向量字段
        anns_field='sparse',
        param={"metric_type": "BM25"},
        # 最大匹配三条
        limit=3
    )

    # 通过数据库客户端查询
    results = milvus_client.hybrid_search(
        # 使用混合检索方法
        collection_name=collection_name,
        # 组和两种请求
        reqs=[dense_query,sparse_query],
        # 设置返回条数
        limit=3,
        # 设置返回的字段
        output_fields=['id', 'h2', 'content'],
        # 设置每种请求的权重
        ranker=ranker
    )

    for hits in results:
        for hit in hits:
            print(json.dumps(hit,indent=2,ensure_ascii=False))

In [12]:
# cross-encoder rerank
from pymilvus import Function, FunctionType

# 指定重排的函数,
def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        name="dashscope_semantic_ranker",      # ranker名词，唯一即可
        input_field_names=["content"],          # 原始文档字段
        function_type=FunctionType.RERANK,  # ranker类型，这里是固定值
        params={
            "reranker": "model",  # rerank类型，基于模型rerank，也就是cross-encoder
            "provider": "ali",  # rerank模型提供者
            "model_name": "gte-rerank-v2",  # rerank模型名称
            # 也就是用户的问题
            "queries": queries,  # 查询条件
            "max_client_batch_size": 5,  # 单批次中处理的文档最大数量
        },
    )

# 基于排序模型实现的ranker重排序
def cross_encoder_hybrid_search(query):
    # 创建排序器对象
    ranker = create_cross_encoder_ranker([query])

    # 调用混合检索的方法完成混合检索
    hybrid_search(user_question=query, ranker=ranker)

In [13]:
# 测试cross_encoder融合
cross_encoder_hybrid_search("教育的负面作用是什么")


{
  "id": 15,
  "distance": 0.29470494389533997,
  "entity": {
    "content": "# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。",
    "h2": "第一节 教育的功能",
    "id": 15
  }
}
{
  "id": 16,
  "distance": 0.1490025371313095,
  "entity": {
    "content": "## 第二节 教育和社会的关系  \n### （一）政治与教育的关系  \n政治（经济制度）对教育具有决定作用，是决定教育性质的直接因素。具体表现为：决定了教育的性质和目的；决定了教育的领导权；决定了哪部分社会成员享有受教育的权利；决定了部分的教育内容；决定了教育的管理体制。  \n教育对政治发展的作用表现在：教育促进人的政治社会化；教育培养现代政治法律人才；教育促进现代政治民主化。",
    "h2": "第二节 教育和社会的关系",
    "id": 16
  }
}
{
  "id": 20,
  "distance": 0.14625170826911926,
  "entity": {
    "content": "### （五）人口与教育的关系  \n人口对教育的作用主要表现在：人口的数量和增长是决定教育事业的规模、速度的一个重要因素；人口的质量影响教育的质量；人口的结构影响着教育的发展；人口流动影响着教育的发展。  \n教育对人口的作用主要表现在：控制人口的数量；提高人口的质量；改善人口结构；调整人才构成与流动等作用。",
    "h2": "第二节 教育和社会的关系",
    "id": 20
  }
}
